# 08 Visualisation and postprocessing

| | |
|---|---|
| **Purpose** | Everything after the solve: field assembly settings, fields on a plane, zooming into a subdomain, fields on arbitrary points, custom plots, and saving and reloading a session. |
| **Inputs** | 200 kHz plane wave along +x; two disjoint spheres in water: fat (radius 7 mm at (-10, 0, 0) mm) and cortical bone (radius 5 mm at (10, 0, 0) mm), element size 2 mm. |
| **Outputs** | Total and scattered pressure plots, a zoomed plot, a point-cloud plot (`outputs/08_visualisation_and_postprocess_*.png`) and an exported session file `outputs/optimus_exported_data*`. |
| **Checks** | GMRES converged; the point-cloud field equals the plane field at the same points; the scattered field equals total minus incident outside the spheres; the exported solution vector reloads unchanged. |
| **Runtime** | About 25 s full, 16 s CI mode (Docker image, 4 CPUs, 9 Nov 2026). CI mode uses smaller grids. |
| **Adapted from** | `notebooks/Visualisation and postprocess.ipynb` (upstream OptimUS). |

## Notes

* `VisualisePlane` builds a regular grid on a plane and classifies each point as inside a domain,
  outside all domains, or on a boundary (within a tolerance), then evaluates the right potential
  operators for each group.
* `VisualiseCloudPoints` does the same for any (3, N) array of points.
* Field assembly can be `dense` (accurate, fine for a few thousand points) or `hmat` (hierarchical
  matrix compression, for many points).
* `export_to_file` pickles parameters, the model's picklable attributes and the postprocess object into
  a `shelve` database. BEM operators are not picklable and must be rebuilt.

> **To expand (Yash):** explain how points on or near a boundary are treated (the solid angle method, `optimus/postprocess/solid_angle_method.py`) and why that matters for plots close to the skull.

## Setup

Imports, the `CI_MODE` switch and the output folder.

In [ ]:
NOTEBOOK_NAME = "08_visualisation_and_postprocess"

%matplotlib inline
import json
import os
import time

import numpy as np
import matplotlib.pyplot as plt

import optimus

# CI_MODE uses smaller settings so the notebook finishes in a few minutes in CI.
# Set it from the shell with OPTIMUS_CI_MODE=1, or change it here by hand.
CI_MODE = os.environ.get("OPTIMUS_CI_MODE", "0") == "1"

# Figures and a small JSON file of key results are written here.
OUTPUT_DIR = os.environ.get("OPTIMUS_OUTPUT_DIR", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS = {"notebook": NOTEBOOK_NAME, "ci_mode": CI_MODE,
           "optimus_version": optimus.__version__, "checks": {}}
START_TIME = time.time()

print("OptimUS", optimus.__version__, "| CI mode:", CI_MODE, "| outputs in:", OUTPUT_DIR)

In [ ]:
def check(name, condition, detail):
    """Record a named check and stop the notebook if it fails."""
    RESULTS["checks"][name] = bool(condition)
    print(("PASS  " if condition else "FAIL  ") + name + ": " + detail)
    assert condition, name + " failed: " + detail


def gmres_converged(model):
    """GMRES stops early (before maxiter) only when the tolerance is reached."""
    if hasattr(model, "linear_solve_iteration_count"):
        iterations = model.linear_solve_iteration_count
    else:
        iterations = model.iteration_count
    return iterations, iterations < optimus.global_parameters.linalg.maxiter


def save_figures(figs, stem):
    """Save a list of matplotlib figures as PNG files in OUTPUT_DIR."""
    if not isinstance(figs, (list, tuple)):
        figs = [figs]
    for i, fig in enumerate(figs):
        fig.savefig(os.path.join(OUTPUT_DIR, "%s_%d.png" % (stem, i + 1)), dpi=110,
                    bbox_inches="tight")

## 1. Global postprocessing parameters

In [ ]:
optimus.global_parameters.bem.set_field_assembly_type("dense")
optimus.global_parameters.bem.print()

## 2. Model

In [ ]:
frequency = 200e3
source = optimus.source.create_planewave(frequency, direction=(1, 0, 0))
geometry_left = optimus.geometry.shapes.Sphere(radius=7e-3, origin=(-10e-3, 0, 0), element_size=2e-3)
geometry_right = optimus.geometry.shapes.Sphere(radius=5e-3, origin=(10e-3, 0, 0), element_size=2e-3)
material_ext = optimus.material.load_material("water")
materials_int = optimus.material.load_material(("fat", "bone (cortical)"))

model = optimus.model.create_acoustic_model(source, (geometry_left, geometry_right),
                                            material_ext, materials_int)
t0 = time.time()
model.solve()
RESULTS["solve_time_s"] = round(time.time() - t0, 1)
iterations, converged = gmres_converged(model)
RESULTS["gmres_iterations"] = int(iterations)
print("GMRES iterations:", iterations, "| solve time %.1f s" % RESULTS["solve_time_s"])

## 3. Field on a plane

`resolution` is the number of points along each plane axis, `plane_axes` picks the axes (0, 1, 2 for
x, y, z), `plane_offset` moves the plane along the third axis, and `bounding_box` is
`(axis1_min, axis1_max, axis2_min, axis2_max)`.

In [ ]:
plane_resolution = (56, 66) if CI_MODE else (111, 131)
plane = optimus.postprocess.VisualisePlane(model)
plane.create_computational_grid(resolution=plane_resolution, plane_axes=(0, 1), plane_offset=0,
                                bounding_box=(-20e-3, 18e-3, -10e-3, 13e-3))
plane.compute_fields()
figs = optimus.postprocess.plot_pressure_field(plane, field="total_pressure", unit="Pa")
save_figures(figs, NOTEBOOK_NAME + "_total")
figs = optimus.postprocess.plot_pressure_field(plane, field="scattered_pressure", unit="Pa", clim=(-2, 2))
save_figures(figs, NOTEBOOK_NAME + "_scattered")

The arrays behind the plots are available directly: `plane.field.total_field` has shape
`(N1 * N2,)` and `plane.total_field_imshow` has shape `(N2, N1)`, ready for `imshow`.
`plane.domains_edges` holds the outline of each domain in the plane.

In [ ]:
fig = plt.figure(figsize=(10, 5))
plt.imshow(np.real(plane.total_field_imshow), extent=plane.bounding_box, clim=(-2, 2),
           cmap="seismic", interpolation="bilinear")
for edge_x, edge_y in plane.domains_edges:
    plt.plot(edge_x, edge_y, color="black", linestyle="dashed", linewidth=2)
plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.colorbar(label="Re(p_total) [Pa]")
save_figures(fig, NOTEBOOK_NAME + "_custom")
plt.show()

## 4. Zoom into a subdomain

A small bounding box gives detail without raising the resolution everywhere.

In [ ]:
zoom = optimus.postprocess.VisualisePlane(model)
zoom.create_computational_grid(resolution=(40, 60) if CI_MODE else (111, 121), plane_axes=(0, 1),
                               plane_offset=0, bounding_box=(4e-3, 7e-3, -5e-3, 5e-3))
zoom.compute_fields()
figs = optimus.postprocess.plot_pressure_field(zoom, field="total_pressure", unit="Pa")
save_figures(figs, NOTEBOOK_NAME + "_zoom")

## 5. Field on arbitrary points

Here the points are the plane's own grid, so the result can be checked against section 3.

In [ ]:
cloud = optimus.postprocess.VisualiseCloudPoints(model)
cloud.create_computational_grid(points=plane.points)
cloud.compute_fields()
cloud_imshow = optimus.postprocess.common.array_to_imshow(cloud.field.total_field.reshape(plane_resolution))
cloud_difference = np.nanmax(np.abs(cloud_imshow - plane.total_field_imshow))
print("Largest difference between cloud and plane fields: %.2e Pa" % cloud_difference)

## 6. Save and reload a session

In [ ]:
export_name = os.path.join(OUTPUT_DIR, "optimus_exported_data")
optimus.postprocess.export_to_file(model=model, post_process=plane,
                                   global_parameters=optimus.global_parameters, file_name=export_name)
exported_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("optimus_exported_data")]
print("Files written:", exported_files)
imported = optimus.postprocess.import_from_file(export_name + ".db")
reloaded_solution = imported["model"].solution_vector
print("First entries of the reloaded solution:", reloaded_solution[:3])

## 7. Checks

In [ ]:
check("gmres_converged", converged, "%d iterations" % iterations)
check("cloud_matches_plane", cloud_difference < 1e-10,
      "max difference %.2e Pa" % cloud_difference)

outside = np.isfinite(plane.field.incident_field) & (np.abs(plane.field.incident_field) > 0)
split_error = np.max(np.abs(plane.field.total_field[outside]
                            - plane.field.scattered_field[outside] - plane.field.incident_field[outside]))
check("total_is_incident_plus_scattered", split_error < 1e-10,
      "max |p_tot - p_sc - p_inc| outside the spheres = %.2e" % split_error)
check("export_roundtrip", np.array_equal(reloaded_solution, model.solution_vector),
      "reloaded solution vector identical (%d entries)" % len(reloaded_solution))

## Summary

Key numbers and check results, also saved as JSON in `OUTPUT_DIR` so CI and the assistant can read them.

In [ ]:
RESULTS["runtime_s"] = round(time.time() - START_TIME, 1)
with open(os.path.join(OUTPUT_DIR, NOTEBOOK_NAME + "_results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2, sort_keys=True)
print(json.dumps(RESULTS, indent=2, sort_keys=True))